# Docling on Dr.DocBench pages
Settings: **Accelerator = GPU**, **Internet = On**, dataset `drdoc-pages-v2` as input.
Saves Docling's full document JSON (layout boxes incl. `picture`, captions, tables, text) and markdown per page.
Run with `LIMIT = 20` first, then `LIMIT = None` (1495 pages).

In [ ]:
!nvidia-smi | head -15
!python --version

In [ ]:
import os, glob, csv, zipfile, shutil, json, time, sys, subprocess
LIMIT = 20            # smoke test: this many pages spread over dev+test. None = full run (1495 pages).
SPLITS = ("dev", "test")
manifests = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
zips = glob.glob("/kaggle/input/**/drdocbench_pages.zip", recursive=True)
if manifests:
    root = os.path.dirname(manifests[0])
elif zips:
    zipfile.ZipFile(zips[0]).extractall("/kaggle/working/data"); root = "/kaggle/working/data"
else:
    root = None
assert root, "dataset not found under /kaggle/input: " + str(glob.glob("/kaggle/input/**", recursive=False))
rows = [r for r in csv.DictReader(open(root + "/manifest.csv")) if r["split"] in SPLITS]
if LIMIT:
    step = max(1, len(rows) // LIMIT); rows = rows[::step][:LIMIT]
stem = lambda r: f"{r['doc']}_{r['page']}"
print(len(rows), "pages to process; first:", rows[0]["file"])

In [ ]:
# retry: package indexes on Kaggle time out now and then
!for i in 1 2 3 4; do python -m pip install -q --timeout 180 --retries 10 docling && break || sleep 30; done
!pip show docling | head -3

In [ ]:
OUT = "/kaggle/working/docling_out"; os.makedirs(OUT, exist_ok=True)
todo = [r for r in rows if not os.path.exists(f"{OUT}/{stem(r)}.json")]
print(len(todo), "pages left")
BUDGET = 9 * 3600     # stop starting new pages after this many seconds, so the notebook finishes and zips cleanly
json.dump([{"file": os.path.abspath(f"{root}/{r['file']}"), "stem": stem(r)} for r in todo], open("/kaggle/working/rows.json", "w"))

WORKER = """
import os, sys, json, time, traceback
shard, n, t_start, BUDGET = int(sys.argv[1]), int(sys.argv[2]), float(sys.argv[3]), float(sys.argv[4])
os.environ["CUDA_VISIBLE_DEVICES"] = str(shard)
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter
try:
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from docling.document_converter import ImageFormatOption
    po = PdfPipelineOptions(); po.do_ocr = True; po.do_table_structure = True
    conv = DocumentConverter(format_options={InputFormat.IMAGE: ImageFormatOption(pipeline_options=po)})
except Exception as e:
    print("shard", shard, "falling back to default converter:", e, flush=True); conv = DocumentConverter()
OUT = "/kaggle/working/docling_out"
items = json.load(open("/kaggle/working/rows.json"))[shard::n]
ok = bad = 0; t0 = time.time()
for i, it in enumerate(items, 1):
    if time.time() - t_start > BUDGET:
        print(f"shard {shard}: time budget reached", flush=True); break
    try:
        doc = conv.convert(it["file"]).document
        json.dump(doc.export_to_dict(), open(f"{OUT}/{it['stem']}.json", "w"))
        open(f"{OUT}/{it['stem']}.md", "w").write(doc.export_to_markdown())
        ok += 1
    except Exception:
        bad += 1; print("shard", shard, "FAILED", it["stem"], traceback.format_exc()[-400:], flush=True)
    if i % 25 == 0 or i == len(items):
        print(f"shard {shard}: {i}/{len(items)} ok={ok} bad={bad} {(time.time() - t0) / i:.1f}s/page", flush=True)
"""
open("/kaggle/working/dworker.py", "w").write(WORKER)

# warm-up in the main process: downloads the layout/table/OCR models once, before the parallel workers start
from docling.document_converter import DocumentConverter
if todo:
    t = time.time(); d = DocumentConverter().convert(os.path.abspath(f"{root}/{todo[0]['file']}")).document
    print(f"warm-up {time.time() - t:.0f}s; markdown head:", d.export_to_markdown()[:300])

NGPU = 2; t_start = time.time()
procs = [subprocess.Popen([sys.executable, "/kaggle/working/dworker.py", str(k), str(NGPU), str(t_start), str(BUDGET)]) for k in range(NGPU)]
while any(p.poll() is None for p in procs):
    time.sleep(120)
    print(f"{time.time() - t_start:.0f}s: {len(glob.glob(OUT + '/*.json'))} pages done", flush=True)
print("exit codes", [p.returncode for p in procs])

In [ ]:
js = sorted(glob.glob(OUT + "/*.json")); print(len(js), "pages")
if js:
    d = json.load(open(js[0])); print("keys:", list(d.keys()))
    print("pictures:", len(d.get("pictures", [])), "texts:", len(d.get("texts", [])), "tables:", len(d.get("tables", [])))
    print("sample text item:", json.dumps(d["texts"][0])[:600] if d.get("texts") else None)

In [ ]:
shutil.make_archive("/kaggle/working/docling_results", "zip", OUT)
print("wrote docling_results.zip", round(os.path.getsize("/kaggle/working/docling_results.zip") / 1e6, 1), "MB")